In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
import zipfile

zip_path = '/content/drive/MyDrive/ML_Skin_Cancer/archive.zip'
extract_path = '/content/dataset'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

In [ ]:
import os, glob
import pandas as pd
import numpy as np
import cv2
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, recall_score, f1_score, classification_report

extract_path = '/content/dataset'
metadata_path = f'{extract_path}/HAM10000_metadata.csv'

df = pd.read_csv(metadata_path)
classes = ['nv','mel','bkl','bcc','akiec','vasc','df']
label2idx = {c:i for i,c in enumerate(classes)}
df['label'] = df['dx'].map(label2idx)

print(df.shape)
df.head()

(10015, 8)


,lesion_id,image_id,dx,dx_type,age,sex,localization,label
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp,2
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp,2
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp,2
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp,2
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear,2


In [ ]:
img_dir1 = f'{extract_path}/HAM10000_images_part_1'
img_dir2 = f'{extract_path}/HAM10000_images_part_2'

all_image_paths = {}
for d in [img_dir1, img_dir2]:
    for f in glob.glob(os.path.join(d, '*.jpg')):
        image_id = os.path.basename(f).replace('.jpg', '')
        all_image_paths[image_id] = f

print(f"Total gambar: {len(all_image_paths)}")

Total gambar: 10015


In [ ]:
train_df, temp_df = train_test_split(df, test_size=0.3, stratify=df['label'], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['label'], random_state=42)

print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

Train: 7010, Val: 1502, Test: 1503


In [ ]:
class HAM10000Dataset(Dataset):
    def __init__(self, df, path_dict, transform=None):
        self.df = df.reset_index(drop=True)
        self.path_dict = path_dict
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = self.path_dict[row['image_id']]
        img = cv2.imread(img_path)

        # Cuma resize, NO DullRazor / CLAHE / Segmentasi
        img = cv2.resize(img, (224, 224))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        if self.transform:
            img = self.transform(img)
        return img, row['label']

In [ ]:
train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(90),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, hue=0.3, saturation=0.3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])

val_test_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])

In [ ]:
train_ds = HAM10000Dataset(train_df, all_image_paths, transform=train_transform)
val_ds   = HAM10000Dataset(val_df, all_image_paths, transform=val_test_transform)
test_ds  = HAM10000Dataset(test_df, all_image_paths, transform=val_test_transform)

class_counts = train_df['label'].value_counts().sort_index().values
class_weights = 1. / class_counts
sample_weights = train_df['label'].map(lambda x: class_weights[x]).values
sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(train_ds, batch_size=32, sampler=sampler)
val_loader   = DataLoader(val_ds, batch_size=32, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=32, shuffle=False)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Pakai device: {device}")

model = models.efficientnet_b0(weights='IMAGENET1K_V1')
model.classifier[1] = nn.Linear(model.classifier[1].in_features, len(classes))
model = model.to(device)

weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=weights_tensor)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

Pakai device: cuda
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 108MB/s] 


In [ ]:
num_epochs = 15

for epoch in range(num_epochs):
    model.train()
    train_loss = 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    model.eval()
    val_loss = 0
    correct, total = 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            preds = torch.argmax(outputs, dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {train_loss/len(train_loader):.4f} | "
          f"Val Loss: {val_loss/len(val_loader):.4f} | Val Acc: {correct/total:.4f}")

torch.save(model.state_dict(), '/content/drive/MyDrive/ML_Skin_Cancer/model_baseline.pth')

Epoch 1/15 | Train Loss: 0.9608 | Val Loss: 1.5076 | Val Acc: 0.1158
Epoch 2/15 | Train Loss: 0.4717 | Val Loss: 1.1575 | Val Acc: 0.1884
Epoch 3/15 | Train Loss: 0.3521 | Val Loss: 1.1475 | Val Acc: 0.2071
Epoch 4/15 | Train Loss: 0.2960 | Val Loss: 0.9361 | Val Acc: 0.3003
Epoch 5/15 | Train Loss: 0.2599 | Val Loss: 0.9338 | Val Acc: 0.3276
Epoch 6/15 | Train Loss: 0.2307 | Val Loss: 0.9536 | Val Acc: 0.3635
Epoch 7/15 | Train Loss: 0.2139 | Val Loss: 0.8465 | Val Acc: 0.4421
Epoch 8/15 | Train Loss: 0.1903 | Val Loss: 0.8905 | Val Acc: 0.4680
Epoch 9/15 | Train Loss: 0.1650 | Val Loss: 0.7851 | Val Acc: 0.5047
Epoch 10/15 | Train Loss: 0.1664 | Val Loss: 0.7624 | Val Acc: 0.5206
Epoch 11/15 | Train Loss: 0.1539 | Val Loss: 0.8072 | Val Acc: 0.5593
Epoch 12/15 | Train Loss: 0.1333 | Val Loss: 0.7635 | Val Acc: 0.5626
Epoch 13/15 | Train Loss: 0.1215 | Val Loss: 0.7468 | Val Acc: 0.5905
Epoch 14/15 | Train Loss: 0.1170 | Val Loss: 0.7243 | Val Acc: 0.6332
Epoch 15/15 | Train Loss: 0.1

In [ ]:
model.eval()
all_preds, all_probs, all_labels = [], [], []

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(device)
        outputs = model(imgs)
        probs = torch.softmax(outputs, dim=1)
        preds = torch.argmax(probs, dim=1)
        all_probs.extend(probs.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

auc = roc_auc_score(all_labels, all_probs, multi_class='ovr')
recall = recall_score(all_labels, all_preds, average='macro')
f1 = f1_score(all_labels, all_preds, average='macro')

print(f"\n=== Hasil Pipeline Baseline (Tanpa Preprocessing) ===")
print(f"AUC-ROC : {auc:.4f}")
print(f"Recall  : {recall:.4f}")
print(f"F1-Score: {f1:.4f}")
print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=classes))


=== Hasil Pipeline Baseline (Tanpa Preprocessing) ===
AUC-ROC : 0.9442
Recall  : 0.7665
F1-Score: 0.5526

Classification Report:
              precision    recall  f1-score   support

          nv       0.99      0.58      0.73      1006
         mel       0.33      0.77      0.46       167
         bkl       0.63      0.58      0.60       165
         bcc       0.52      0.88      0.65        77
       akiec       0.37      0.84      0.52        49
        vasc       0.22      0.95      0.36        22
          df       0.42      0.76      0.54        17

    accuracy                           0.63      1503
   macro avg       0.50      0.77      0.55      1503
weighted avg       0.81      0.63      0.67      1503

